<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 🧭 EarthDaily Agriculture — Weather spatial grouping & cache benchmark

Weather is queried by field **centroid**, so fields whose centroids land in the same
geohash cell get identical answers. `spatial_grouping=True` exploits that twice over:

1. **One call per cell** instead of one per field.
2. **One call per cell, not per window** — dates leave the group key, the cell is pulled
   once over the *union* of its members' windows, and each field is sliced out of that
   single response.
3. **A geohash-keyed cache** — the wide pull is stored per cell, so any later run touching
   that cell is served from disk whatever field set or sub-window it asks for.

Because members are cut back to their own window, the grouped result is **identical to an
ungrouped run**. This notebook measures the saving and proves that equality on live data.

> **On cumulative columns.** Whether a widened window changes values depends on the
> parameter, and the column *name* is not a reliable guide — this was measured against
> prod by requesting one field over a wide and a narrow window and diffing the overlap:
> 
> | Column | Overlap identical | Window-dependent? |
> |---|---|---|
> | `precipitation.cumulative` | 62/62, max diff 0.0 | **No** — it is a per-day total |
> | `temperature.standardMax` | 62/62, max diff 0.0 | No |
> | `daily_gdd` | 62/62, max diff 0.0 | No |
> | `cumulated_gdd` | 0/62, constant offset 977.2 | **Yes** — accumulates from request start |
> 
> So WeatherExtractor declares **no** cumulative columns and passes values through
> untouched, while GDDExtractor declares `cumulated_gdd` and re-zeroes it per member.
> Rebasing is always explicit per extractor, never inferred from the name.

| Run | Grouping | Cache | What it shows |
|---|---|---|---|
| **A** | off | off | Baseline: one API call per field |
| **B** | on | off | Dedup + window union only |
| **C** | on | on (cold) | Same calls as B, but results persisted per cell |
| **D** | on | on (warm) | Zero API calls — everything served from the cell cache |

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
import time

import pandas as pd

from earthdaily.agriculture.core.geometry import centroid_geohash
from earthdaily.agriculture.extractors.weather_functions import WeatherExtractor
from earthdaily.agriculture.services.workflow_manager import WorkflowManager

manager = WorkflowManager("prod", log_to_console=False, log_level="INFO")

extractor = WeatherExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

# One per-day parameter and one cumulative parameter: the cumulative one is what
# window widening would corrupt if it were not rebased, so it is the real test.
extractor.setup_weather_parameters(
    weather_type="HISTORICAL_DAILY",
    weather_parameters=["Temperature.standardmax", "precipitation.cumulative"],
)
print("weather params:", extractor.weather_params)

## **🛠️ Step 2: Build a clustered field set**

Three clusters of five fields. Within a cluster the fields sit a few hundred metres apart
— same geohash-5 cell (~4.9 km) — but they deliberately carry **different date windows**,
which is the case the old date-in-the-key grouping could not collapse.

In [ ]:
CLUSTERS = {
    "cell_a": (-93.500, 42.000),
    "cell_b": (-93.220, 41.780),
    "cell_c": (-94.100, 42.310),
}
FIELDS_PER_CLUSTER = 5

# Staggered, overlapping windows — all well inside the 400-day widening cap.
WINDOWS = [
    ("2025-04-01", "2025-07-01"),
    ("2025-05-01", "2025-08-01"),
    ("2025-04-15", "2025-07-15"),
]


def square(lon, lat, size=0.002):
    """Small square polygon (~200 m) centred on (lon, lat)."""
    return (
        f"POLYGON(({lon:.6f} {lat:.6f}, {lon + size:.6f} {lat:.6f}, "
        f"{lon + size:.6f} {lat + size:.6f}, {lon:.6f} {lat + size:.6f}, "
        f"{lon:.6f} {lat:.6f}))"
    )


rows = []
for cluster, (lon0, lat0) in CLUSTERS.items():
    for i in range(FIELDS_PER_CLUSTER):
        start, end = WINDOWS[i % len(WINDOWS)]
        rows.append(
            {
                "id": f"{cluster}_f{i}",
                "geometry": square(lon0 + 0.004 * i, lat0 + 0.003 * i),
                "start_date": start,
                "end_date": end,
            }
        )

entities = pd.DataFrame(rows)
entities["geohash"] = entities["geometry"].map(lambda g: centroid_geohash(g, precision=5))

print(f"{len(entities)} fields across {entities['geohash'].nunique()} geohash cell(s) "
      f"and {entities[['start_date', 'end_date']].drop_duplicates().shape[0]} distinct windows")
entities.groupby("geohash").agg(fields=("id", "count"), windows=("start_date", "nunique"))

## **📥 Step 3: The four runs**

`skip_export=True` everywhere — we are measuring calls, not producing deliverables.

In [ ]:
RESULTS = {}


def run(label, **kwargs):
    """Run a bulk extraction, timing it and recording how many API calls it made."""
    print(f"\n{'=' * 70}\n▶ Run {label}\n{'=' * 70}")
    t0 = time.perf_counter()
    out = extractor.process_entity_weather_bulk_parallel(
        entity_list=entities.drop(columns=["geohash"]),
        skip_export=True,
        **kwargs,
    )
    elapsed = time.perf_counter() - t0

    df = out.get("results_df", pd.DataFrame())
    # Ungrouped runs make one call per field; grouped runs report theirs.
    calls = out.get("representative_calls", len(entities))
    RESULTS[label] = {
        "api_calls": calls,
        "seconds": round(elapsed, 1),
        "rows": len(df),
        "entities": int(df["entity_id"].nunique()) if "entity_id" in df.columns else 0,
        "cache_hit": out.get("cache_hit"),
        "df": df,
    }
    print(f"→ {calls} API call(s), {elapsed:.1f}s, {len(df)} rows")
    return df

### Run A — baseline: no grouping, no cache

In [ ]:
df_a = run("A", spatial_grouping=False, use_cache=False)
df_a.head()

### Run B — grouping on, cache off

One call per cell, over the union of that cell's windows.

In [ ]:
df_b = run("B", spatial_grouping=True, use_cache=False)
df_b.head()

### Run C — grouping + cache, cold

Same work as B, but each cell's wide pull is written to the geohash-keyed cache.

In [ ]:
# Start from an empty cell cache so 'cold' really is cold.
extractor.clear_spatial_cache(extractor.weather_params)

df_c = run("C", spatial_grouping=True, use_cache=True)
extractor.spatial_cache_info(extractor.weather_params)

### Run D — grouping + cache, warm

The same request again. Every cell is already covered, so nothing should reach the API.

In [ ]:
df_d = run("D", spatial_grouping=True, use_cache=True)
df_d.head()

### Run E — a *different* field set in the same cells

This is what a cell-keyed cache buys over an entity-keyed one: fields never seen before,
asking for a narrower window inside an already-cached cell, cost no API call.

In [ ]:
newcomers = pd.DataFrame(
    [
        {
            "id": f"newcomer_{name}",
            "geometry": square(lon + 0.001, lat + 0.001),
            "start_date": "2025-05-10",
            "end_date": "2025-06-20",
        }
        for name, (lon, lat) in CLUSTERS.items()
    ]
)

t0 = time.perf_counter()
out_e = extractor.process_entity_weather_bulk_parallel(
    entity_list=newcomers, skip_export=True, spatial_grouping=True, use_cache=True
)
print(f"→ {out_e['representative_calls']} API call(s) for {len(newcomers)} unseen fields "
      f"in {time.perf_counter() - t0:.1f}s")
out_e["results_df"].head()

## **🔍 Step 4: Equivalence — the check that matters**

Grouping is only worth having if it changes nothing but the call count. Every grouped run
must match the ungrouped baseline row for row, including the cumulative column that the
widened window would otherwise inflate.

In [ ]:
KEY = ["entity_id", "date"]


def compare(baseline, other, label):
    """Assert a grouped result is identical to the ungrouped baseline."""
    cols = [c for c in baseline.columns if c in other.columns]
    a = baseline[cols].sort_values(KEY).reset_index(drop=True)
    b = other[cols].sort_values(KEY).reset_index(drop=True)
    try:
        pd.testing.assert_frame_equal(a, b, check_dtype=False, rtol=1e-6)
        print(f"✅ {label}: identical to baseline ({len(a)} rows, {len(cols)} columns)")
        return True
    except AssertionError as exc:
        print(f"❌ {label}: DIFFERS from baseline\n{exc}")
        return False


for label, df in [("B (grouped)", df_b), ("C (grouped+cache cold)", df_c), ("D (grouped+cache warm)", df_d)]:
    compare(df_a, df, label)

### Why weather needs no value correction

Re-derive it here rather than taking it on trust: ask for **one** field over a wide and a
narrow window, then diff the overlapping dates. Identical values mean the parameter is
per-day and a widened group window cannot distort it.

In [ ]:
CUM = "precipitation.cumulative"


def probe(start, end):
    one = entities.head(1).drop(columns=["gh5"]).copy()
    one["start_date"], one["end_date"] = start, end
    out = extractor.process_entity_weather_bulk_parallel(
        entity_list=one, skip_export=True, spatial_grouping=False, use_cache=False
    )
    d = out["results_df"].copy()
    d["date"] = pd.to_datetime(d["date"], utc=True).dt.tz_localize(None)
    return d.sort_values("date").reset_index(drop=True)


wide = probe("2025-05-01", "2025-08-31")
narrow = probe("2025-07-01", "2025-08-31")
overlap = wide.merge(narrow, on="date", suffixes=("_wide", "_narrow"))

for col in [CUM, "temperature.standardMax"]:
    if f"{col}_wide" not in overlap.columns:
        continue
    diff = (overlap[f"{col}_wide"] - overlap[f"{col}_narrow"]).abs()
    verdict = "per-day (safe to broadcast)" if diff.max() < 1e-9 else "WINDOW-DEPENDENT"
    print(f"{col:28s} identical {int((diff < 1e-9).sum())}/{len(overlap)}, max diff {diff.max():.3f} -> {verdict}")

# Monotonic would suggest a running total; it is not.
print(f"\n{CUM} monotonically increasing within one request? {wide[CUM].is_monotonic_increasing}")
print(f"first 8 values: {list(wide[CUM].head(8))}")

## **📊 Step 5: Summary**

In [ ]:
summary = pd.DataFrame(
    {
        label: {
            "API calls": r["api_calls"],
            "Seconds": r["seconds"],
            "Rows": r["rows"],
            "Entities": r["entities"],
            "Cells from cache": r["cache_hit"],
        }
        for label, r in RESULTS.items()
    }
).T

baseline_calls = RESULTS["A"]["api_calls"]
summary["Calls saved vs A"] = baseline_calls - summary["API calls"]
summary["Reduction"] = (summary["Calls saved vs A"] / baseline_calls * 100).round(0).astype(int).astype(str) + "%"
summary

### Notes & caveats

- **Precision.** `spatial_precision=5` (~4.9 km) matches the weather grid. Raise it to 6
  (~1.2 km) for a more conservative bucket and less dedup.
- **Widening cap.** `spatial_max_window_days` (default 400) stops one long-history field
  from dragging its whole cell into a decade-long pull — such a cell is split instead.
- **Cache coverage is all-or-nothing per cell.** A cell whose cached dates do not span the
  requested window is refetched in full rather than patched, which keeps the stored series
  contiguous. Widen once, reuse often.
- **GDD** groups and widens the same way, but *does* need the value correction:
  `cumulated_gdd` accumulates from the request start (measured: a constant 977.2 offset
  between a wide and a narrow window), so it is re-zeroed per member — year by year when
  `reset_cumulative_every_year` is set. `daily_gdd` is per-day and passes through.
- **GDD-offset does not participate in window widening** — it returns one row per entity
  from a single base date, so there is no series to slice; it still groups by geohash on
  exact matches.
- **Adding a new parameter?** Re-run the wide-vs-narrow probe above before assuming it is
  safe. A name containing 'cumulative' proved nothing either way.
- Clear caches between benchmark sessions with `extractor.clear_spatial_cache(extractor.weather_params)`.